# AI-Enabled Concrete Performance Passports for OpenBIM
## Unified, Google-Drive-backed, checkpointed training notebook

This single notebook runs the **entire** project end-to-end:

1. **Data** — download + clean the UCI concrete strength dataset (Yeh, 1998) and the public ready-mix EPD dataset (Broyles et al., 2024).
2. **Strength model** — train & compare regressors (RF / GBR / ExtraTrees / Linear, + XGBoost/LightGBM/CatBoost if available); pick the best.
3. **Explainability** — permutation / SHAP feature importance.
4. **Carbon** — ICE-factor GWP estimate + EPD-benchmarked carbon class.
5. **Passports** — generate Concrete Performance Passports (CSV + JSON + per-element JSON).
6. **OpenBIM / IDS** — IFC mapping, proposed Pset, IDS requirement matrix + conceptual IDS XML.
7. **Figures / tables / verification**.

### Two things make this notebook robust
- **Everything is saved to your Google Drive.** All data, models, figures, tables and passports are written under a project folder on Drive, so nothing is lost when the Colab runtime disconnects.
- **Checkpointing / resume.** Each phase records a checkpoint on Drive. If the runtime stops, just **re-run the notebook from the top** — completed phases are detected and skipped, and training continues where it left off.


## How resume works
- A file `_checkpoints.json` is kept in the Drive project folder.
- Before running a phase, the notebook checks (a) the checkpoint flag **and** (b) that the phase's expected output files actually exist on Drive.
- If both are satisfied the phase is **skipped**; otherwise it runs and re-records its checkpoint.
- To force a phase (or everything) to re-run, use the **reset helper** near the bottom.

> Tip: For the strength model you can enable a GPU runtime (Runtime -> Change runtime type) to speed up XGBoost, but CPU is perfectly fine — the full run takes only a few minutes.


## 1. Mount Google Drive
This is where all artefacts are persisted. Outside Colab, the notebook falls back to a local folder so it still runs.

In [ ]:
import os, sys, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

DRIVE_MOUNT = "/content/drive"
if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=False)
    DRIVE_ROOT = Path(DRIVE_MOUNT) / "MyDrive"
else:
    # Local / non-Colab fallback: persist to a folder in the home directory.
    DRIVE_ROOT = Path(os.environ.get("CPP_LOCAL_DRIVE", Path.home() / "cpp_drive"))
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Not running in Colab -> persisting to local folder:", DRIVE_ROOT)

assert DRIVE_ROOT.exists(), f"Drive root not found: {DRIVE_ROOT}"
print("Drive root:", DRIVE_ROOT)

## 2. Settings, clone the code, install dependencies
- `PROJECT_NAME` is the folder created on your Drive that holds **all** outputs.
- The repository **code** is cloned to fast local storage each session (cheap, re-clonable); the **data/models/outputs** live on Drive (expensive, persistent).

In [ ]:
# ---- Settings you may edit -------------------------------------------------
PROJECT_NAME = "ConcretePerformancePassport"   # Drive folder for all artefacts
REPO_URL     = "https://github.com/sayyedalimrj/Paper.git"
REPO_BRANCH  = "main"
N_PASSPORTS  = 30          # number of demonstration passports to generate
USE_SHAP     = False       # True = SHAP explanations (slower); False = permutation importance
FORCE_OFFLINE = False      # True = never hit the network (uses cached data only)
# ---------------------------------------------------------------------------

# Persistent project root on Drive: all data/ and outputs/ go here.
PROJECT_ROOT = DRIVE_ROOT / PROJECT_NAME
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent project root (Drive):", PROJECT_ROOT)

# Local clone location for the source code.
if IN_COLAB:
    CODE_DIR = Path("/content/Paper")
else:
    # If we are already inside the repo, use it; else clone next to cwd.
    here = Path.cwd()
    CODE_DIR = here if (here / "src" / "config.py").exists() else here / "Paper"

if not (CODE_DIR / "src" / "config.py").exists():
    print("Cloning repository ->", CODE_DIR)
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CODE_DIR)], check=True)
else:
    print("Source code already present ->", CODE_DIR)

# Make the `src` package importable.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Install dependencies (quiet). Most are pre-installed in Colab.
req = CODE_DIR / "requirements.txt"
if req.exists():
    print("Installing dependencies from", req, "...")
    os.system(f"pip install -q -r '{req}' 2>/dev/null")
else:
    os.system("pip install -q numpy pandas scipy scikit-learn xgboost joblib matplotlib requests openpyxl xlrd lxml jinja2 tabulate pyyaml 2>/dev/null")
print("Dependencies ready.")

## 3. Point the project at Drive and import the modules
`CPP_PROJECT_ROOT` is read by `src/config.py` **at import time**, so it must be set *before* importing. This redirects every `data/` and `outputs/` path onto your Drive folder.

In [ ]:
# Redirect ALL data/outputs onto Drive BEFORE importing config.
os.environ["CPP_PROJECT_ROOT"] = str(PROJECT_ROOT)
if FORCE_OFFLINE:
    os.environ["USE_CACHED_ONLY"] = "1"

from src import (config, utils, data_download, data_cleaning,
                 train_strength_model, explainability, train_carbon_model,
                 generate_passports, ifc_mapping, pipeline)

config.ensure_dirs()
utils.set_seed()

print("config.PROJECT_ROOT :", config.PROJECT_ROOT)
print("data ->", config.DATA_DIR)
print("outputs ->", config.OUTPUTS_DIR)
assert str(config.PROJECT_ROOT) == str(PROJECT_ROOT), "Project root not redirected to Drive!"
print("\nAll artefacts will be saved on Drive at:", PROJECT_ROOT)

## 4. Checkpoint manager (enables resume)
`run_phase(name, fn, outputs=[...])` runs a phase only if it has not already completed **and** produced its outputs. Results are cached in-memory per session too.

In [ ]:
CKPT_PATH = PROJECT_ROOT / "_checkpoints.json"

class Checkpoints:
    def __init__(self, path):
        self.path = Path(path)
        self.state = {}
        if self.path.exists():
            try:
                self.state = json.loads(self.path.read_text())
            except Exception:
                self.state = {}
        self._cache = {}

    def _save(self):
        self.path.write_text(json.dumps(self.state, indent=2, default=str))

    def is_done(self, name, outputs):
        if not self.state.get(name, {}).get("done"):
            return False
        # Verify the recorded output files still exist on Drive.
        for o in outputs or []:
            if not Path(o).exists():
                return False
        return True

    def run_phase(self, name, fn, outputs=None, force=False):
        outputs = [str(o) for o in (outputs or [])]
        if not force and self.is_done(name, outputs):
            print(f"[SKIP] '{name}' already complete (checkpoint + outputs present).")
            return self._cache.get(name)
        print(f"[RUN ] '{name}' ...")
        t0 = time.time()
        result = fn()
        dt = time.time() - t0
        missing = [o for o in outputs if not Path(o).exists()]
        if missing:
            raise RuntimeError(f"Phase '{name}' finished but expected outputs are missing: {missing}")
        self.state[name] = {"done": True, "seconds": round(dt, 2),
                            "finished_utc": utils.utc_timestamp(), "outputs": outputs}
        self._save()
        self._cache[name] = result
        print(f"[DONE] '{name}' in {dt:.1f}s  ({len(outputs)} output(s) verified)")
        return result

    def reset(self, name=None):
        if name is None:
            self.state = {}
            print("Reset ALL checkpoints (outputs on Drive are kept; re-running will overwrite them).")
        else:
            self.state.pop(name, None)
            print(f"Reset checkpoint: {name}")
        self._save()

    def status(self):
        import pandas as pd
        if not self.state:
            print("No checkpoints recorded yet.")
            return
        rows = [{"phase": k, "done": v.get("done"), "seconds": v.get("seconds"),
                 "finished_utc": v.get("finished_utc")} for k, v in self.state.items()]
        return pd.DataFrame(rows)

ckpt = Checkpoints(CKPT_PATH)
print("Checkpoint file:", CKPT_PATH)
display(ckpt.status())

## 5. (Optional) Source availability check
A quick reachability probe of the public sources. Safe to skip if offline.

In [ ]:
if not FORCE_OFFLINE:
    import requests
    SOURCES = {
        "UCI .xls": "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls",
        "Mendeley EPD API": "https://data.mendeley.com/public-api/datasets/r4jgxk2mhn/files?folder_id=root&version=3",
    }
    for name, url in SOURCES.items():
        try:
            r = requests.head(url, timeout=20, allow_redirects=True)
            print(f"  {name}: reachable (HTTP {r.status_code})")
        except Exception as e:
            print(f"  {name}: not reachable now ({type(e).__name__}) — fallbacks will be used.")
else:
    print("FORCE_OFFLINE=True -> skipping network checks; cached data will be used.")

## 6. Phase 1 — Data: download + clean
Downloads the UCI strength dataset (with OpenML fallback) and writes the cleaned, feature-engineered CSV.

In [ ]:
import pandas as pd

def _phase_data():
    data_download.download_concrete_strength()
    clean_path = data_cleaning.clean()
    df = pd.read_csv(clean_path)
    ds = pipeline.dataset_summary(df)
    return {"clean_path": str(clean_path), "summary": ds, "n_rows": len(df)}

data_res = ckpt.run_phase(
    "data",
    _phase_data,
    outputs=[config.PROCESSED_DIR / "concrete_strength_clean.csv",
             config.TABLES_DIR / "dataset_summary.csv"],
)

# Always (re)load the cleaned frame for later cells, even on skip.
clean_csv = config.PROCESSED_DIR / "concrete_strength_clean.csv"
df_clean = pd.read_csv(clean_csv)
print("Cleaned dataset:", df_clean.shape)
display(df_clean.head())
display(pd.read_csv(config.TABLES_DIR / "dataset_summary.csv"))

## 7. Phase 2 — Train the compressive-strength model
Trains and compares all available regressors, evaluates on a hold-out split + 5-fold CV, and persists the **best** model (with an uncertainty model + residual std) to Drive.

In [ ]:
def _phase_strength():
    res = train_strength_model.train(clean_csv)
    pipeline.model_performance_summary(res["metrics"], res["best_name"])
    return {"best_name": res["best_name"], "features": res["features"]}

strength_res = ckpt.run_phase(
    "strength_model",
    _phase_strength,
    outputs=[config.MODELS_DIR / "best_strength_model.pkl",
             config.TABLES_DIR / "strength_model_metrics.csv",
             config.FIGURES_DIR / "strength_model_comparison.png"],
)

metrics = pd.read_csv(config.TABLES_DIR / "strength_model_metrics.csv")
print("Model comparison (sorted by hold-out R2):")
display(metrics)
import joblib
_art = joblib.load(config.MODELS_DIR / "best_strength_model.pkl")
print("Best model:", _art["model_name"], "| trained:", _art["trained_utc"])

## 8. Phase 3 — Explainability
Feature importance for the best model (permutation by default; set `USE_SHAP=True` in settings for SHAP).

In [ ]:
def _phase_explain():
    art = joblib.load(config.MODELS_DIR / "best_strength_model.pkl")
    X = df_clean[art["features"]].astype(float)
    y = df_clean[config.STRENGTH_TARGET].astype(float)
    imp, method = explainability.explain(art["model"], X, art["features"], y, use_shap=USE_SHAP)
    return {"method": method}

explain_res = ckpt.run_phase(
    "explainability",
    _phase_explain,
    outputs=[config.TABLES_DIR / "strength_feature_importance_values.csv",
             config.FIGURES_DIR / "strength_explainability.png"],
)
display(pd.read_csv(config.TABLES_DIR / "strength_feature_importance_values.csv"))

## 9. Phase 4 — Carbon: EPD benchmark + ICE estimate
Downloads the compiled EPD dataset (~80 MB; cached to Drive so it only downloads once), builds per-strength-class GWP benchmarks, and writes ICE per-constituent factors.

In [ ]:
def _phase_carbon():
    r = train_carbon_model.run()
    return {"source_note": r["source_note"]}

carbon_res = ckpt.run_phase(
    "carbon",
    _phase_carbon,
    outputs=[config.EXTERNAL_DIR / "ice_carbon_factors.csv",
             config.TABLES_DIR / "carbon_benchmark_summary.csv",
             config.FIGURES_DIR / "carbon_distribution.png"],
)
print("EPD source:", (carbon_res or {}).get("source_note", "(skipped — already done)"))
display(pd.read_csv(config.TABLES_DIR / "carbon_benchmark_summary.csv"))

## 10. Phase 5 — Generate Concrete Performance Passports
Combines mix/test data, ML prediction + approximate prediction interval + risk class, ICE/EPD carbon class, a rule-based QA/QC decision, and an evidence/provenance layer. Writes a CSV, a combined JSON (with JSON Schema), and one JSON per element.

In [ ]:
def _phase_passports():
    out = generate_passports.generate(n=N_PASSPORTS)
    return {"csv": str(out["csv"]), "n": out["n"]}

pp_res = ckpt.run_phase(
    "passports",
    _phase_passports,
    outputs=[config.TABLES_DIR / "concrete_performance_passports.csv",
             config.TABLES_DIR / "concrete_performance_passports.json",
             config.FIGURES_DIR / "risk_class_distribution.png"],
)
passports = pd.read_csv(config.TABLES_DIR / "concrete_performance_passports.csv")
print("Passports:", passports.shape, "| per-element JSON files:",
      len(list(config.PASSPORTS_DIR.glob("*.json"))))
display(passports[["element_id","element_type","required_strength_mpa","predicted_strength_mpa",
                   "risk_class","qa_qc_decision","carbon_class","evidence_level"]].head(12))

## 11. Phase 6 — OpenBIM / IDS mapping
Passport-field to IFC mapping, the proposed `Pset_ConcretePerformancePassport`, an IDS requirement matrix, and a conceptual IDS XML. `ifcopenshell` is optional (graceful fallback).

In [ ]:
def _phase_openbim():
    r = ifc_mapping.build_mapping()
    return {"ifc_note": r["ifc_note"]}

bim_res = ckpt.run_phase(
    "openbim",
    _phase_openbim,
    outputs=[config.TABLES_DIR / "openbim_mapping.csv",
             config.TABLES_DIR / "ids_requirement_matrix.csv",
             config.TABLES_DIR / "Pset_ConcretePerformancePassport.csv",
             config.CHAPTER_DIR / "concrete_performance_passport.ids.xml"],
)
print("IFC note:", (bim_res or {}).get("ifc_note", "(skipped — already done)"))
display(pd.read_csv(config.TABLES_DIR / "ids_requirement_matrix.csv").head(13))

## 12. Phase 7 — Framework diagram + limitations table

In [ ]:
def _phase_figures():
    pipeline.framework_diagram()
    pipeline.limitations_table()
    return {}

fig_res = ckpt.run_phase(
    "figures_tables",
    _phase_figures,
    outputs=[config.FIGURES_DIR / "framework_diagram.png",
             config.TABLES_DIR / "limitations_and_future_work.csv"],
)
display(pd.read_csv(config.TABLES_DIR / "limitations_and_future_work.csv"))

## 13. Verify all artefacts & preview figures
Confirms every expected file exists on Drive and renders the key figures inline.

In [ ]:
from IPython.display import Image, display as idisplay

EXPECTED = {
    "Cleaned data": config.PROCESSED_DIR / "concrete_strength_clean.csv",
    "Best model": config.MODELS_DIR / "best_strength_model.pkl",
    "Strength metrics": config.TABLES_DIR / "strength_model_metrics.csv",
    "Feature importance": config.TABLES_DIR / "strength_feature_importance_values.csv",
    "Carbon benchmark": config.TABLES_DIR / "carbon_benchmark_summary.csv",
    "ICE factors": config.EXTERNAL_DIR / "ice_carbon_factors.csv",
    "Passports CSV": config.TABLES_DIR / "concrete_performance_passports.csv",
    "Passports JSON": config.TABLES_DIR / "concrete_performance_passports.json",
    "OpenBIM mapping": config.TABLES_DIR / "openbim_mapping.csv",
    "IDS matrix": config.TABLES_DIR / "ids_requirement_matrix.csv",
    "Proposed Pset": config.TABLES_DIR / "Pset_ConcretePerformancePassport.csv",
    "IDS XML": config.CHAPTER_DIR / "concrete_performance_passport.ids.xml",
}
print("=== Artefact verification (on Drive) ===")
all_ok = True
for label, p in EXPECTED.items():
    ok = Path(p).exists()
    all_ok &= ok
    size = (Path(p).stat().st_size if ok else 0)
    print(f"  [{'OK' if ok else 'MISSING'}] {label:<20} {p}  ({size:,} bytes)")
print("\nN per-element passport JSON:", len(list(config.PASSPORTS_DIR.glob('*.json'))))
print("ALL ARTEFACTS PRESENT" if all_ok else "SOME ARTEFACTS MISSING — re-run the relevant phase.")

print("\n=== Key figures ===")
for fig in ["framework_diagram.png", "strength_model_comparison.png",
            "strength_feature_importance.png", "strength_explainability.png",
            "carbon_distribution.png", "risk_class_distribution.png"]:
    fp = config.FIGURES_DIR / fig
    if fp.exists():
        print(fig)
        idisplay(Image(filename=str(fp)))

## 14. Checkpoint status & reset helpers
Show progress, or force phases to re-run.

In [ ]:
# Current checkpoint status:
display(ckpt.status())

# --- To force a re-run, uncomment one of these and run, then re-run the phase cell ---
# ckpt.reset("strength_model")   # re-train only the strength model
# ckpt.reset("carbon")           # re-run carbon (note: keeps the cached 80MB EPD download)
# ckpt.reset()                   # reset EVERYTHING (re-runs all phases on next execution)

## Done
All data, models, figures, tables and passports are saved under your Drive folder
**`MyDrive/<PROJECT_NAME>/`** (`data/` and `outputs/`). Because state is on Drive and
each phase is checkpointed, you can safely close the notebook and re-run it later to
continue exactly where you left off.

**Headline result:** the strength model reaches roughly R2 ~ 0.92 on a hold-out split
(XGBoost when available; otherwise ExtraTrees/RandomForest), with per-sample uncertainty
used to drive the passport risk classes and QA/QC decisions.
